In [2]:
import os
import joblib
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from sklearn.utils import resample
from xgboost import XGBClassifier

In [4]:
os.makedirs("models", exist_ok=True)

DATA_PATH = "cleaned_EMI_dataset.csv"

print("Loading dataset...", DATA_PATH)

df = pd.read_csv(DATA_PATH, low_memory=False)

print("Dataset shape:", df.shape)
df.head()

Loading dataset... cleaned_EMI_dataset.csv
Dataset shape: (404800, 44)


,age,monthly_salary,years_of_employment,monthly_rent,family_size,dependents,school_fees,college_fees,travel_expenses,groceries_utilities,...,company_type_Small,company_type_Startup,house_type_Own,house_type_Rented,emi_scenario_Education EMI,emi_scenario_Home Appliances EMI,emi_scenario_Personal Loan EMI,emi_scenario_Vehicle EMI,emi_eligibility_High_Risk,emi_eligibility_Not_Eligible
0,38.0,82600.0,0.9,20000.0,3,2,0.0,0.0,7200.0,19500.0,...,False,False,False,True,False,False,True,False,False,True
1,38.0,21500.0,7.0,0.0,2,1,5100.0,0.0,1400.0,5400.0,...,False,False,False,False,False,False,False,False,False,True
2,38.0,86100.0,5.8,0.0,4,3,0.0,0.0,10200.0,19400.0,...,False,True,True,False,True,False,False,False,False,False
3,58.0,66800.0,2.2,0.0,5,4,11400.0,0.0,6200.0,11900.0,...,False,False,True,False,False,False,False,True,False,False
4,48.0,57300.0,3.4,0.0,4,3,9400.0,21300.0,3600.0,16200.0,...,False,False,False,False,False,True,False,False,False,True


In [5]:
ohe_cols = ["emi_eligibility_High_Risk", "emi_eligibility_Not_Eligible"]

if not all(c in df.columns for c in ohe_cols):
    raise KeyError(f"Missing one-hot target columns. Expected: {ohe_cols}. Found: {df.columns.tolist()}")

print("Target columns confirmed.")

Target columns confirmed.


In [6]:
df["target_not_eligible"] = df["emi_eligibility_Not_Eligible"].astype(int)

print("Target distribution:")
print(df["target_not_eligible"].value_counts())

Target distribution:
target_not_eligible
1    312868
0     91932
Name: count, dtype: int64


In [7]:
drop_cols = ohe_cols + ["target_not_eligible", "max_monthly_emi"]

features = [c for c in df.columns if c not in drop_cols]

X = df[features].copy()
y = df["target_not_eligible"].copy()

print("Feature count:", len(features))

Feature count: 41


In [8]:
df_full = pd.concat([X, y], axis=1)

major_class = df_full[df_full["target_not_eligible"] == df_full["target_not_eligible"].mode()[0]]
minor_class = df_full[df_full["target_not_eligible"] != df_full["target_not_eligible"].mode()[0]]

if len(major_class) < len(minor_class):
    majority = minor_class
    minority = major_class
else:
    majority = major_class
    minority = minor_class

major_down = resample(
    majority,
    replace=False,
    n_samples=len(minority),
    random_state=42
)

df_balanced = pd.concat([major_down, minority])

df_balanced = df_balanced.sample(frac=1.0, random_state=42).reset_index(drop=True)

X_bal = df_balanced.drop(columns=["target_not_eligible"])
y_bal = df_balanced["target_not_eligible"]

print("Balanced class distribution:")
print(y_bal.value_counts())

Balanced class distribution:
target_not_eligible
1    91932
0    91932
Name: count, dtype: int64


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X_bal,
    y_bal,
    test_size=0.2,
    random_state=42,
    stratify=y_bal
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (147091, 41)
Test shape: (36773, 41)


In [10]:
print("Training XGBoost classifier...")

clf = XGBClassifier(
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=42
)

clf.fit(X_train, y_train)

Training XGBoost classifier...


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [11:27:40] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, ...)

In [11]:
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:,1]

In [12]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("Evaluation Metrics")
print("-------------------")

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)
print("ROC AUC  :", roc_auc)

Evaluation Metrics
-------------------
Accuracy : 0.9684823103907758
Precision: 0.9821438567030507
Recall   : 0.9543130642880453
F1 Score : 0.9680284681801882
ROC AUC  : 0.9962521746883548


In [13]:
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix")
print(cm)

Confusion Matrix
[[18068   319]
 [  840 17546]]


In [14]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.96      0.98      0.97     18387
           1       0.98      0.95      0.97     18386

    accuracy                           0.97     36773
   macro avg       0.97      0.97      0.97     36773
weighted avg       0.97      0.97      0.97     36773

